# Chefer et al. CVPR'21 — code-exact reconstruction with zennit rules

Reconstructs the six ``transformer_attribution`` heatmaps from the reference
implementation's ``example.ipynb`` (``../chefer-transformer-interpretability``,
commit c3e578f) using our zennit LRP rules and composites on a stock timm
``vit_base_patch16_224`` (same ``jx_vit_base_p16_224-80ecf9dd`` checkpoint).

**Reference tensors** are pre-extracted by ``extract_reference_attributions.py``
in the chefer venv and saved as NPZ under ``data/chefer_reference/`` — no
cross-directory imports, no preprocessing drift (the input tensor travels in
the NPZ).

**Code-exact rules** (``zennit_extensions/rules/chefer2021.py``, replacing the
stale paper-faithful variants). Both rules subclass the AttnLRP kernels and
override only the backward with Chefer's ``safe_divide`` stabilizer:
- Matmul: plain z-rule + ÷2 (their ``cam /= 2``, NOT the paper's Eq. 9).
- Add: z-rule + global absolute-mass renormalization (their ``Add`` layer).
- Linear: ``ZPlus(zero_params=['bias'])`` (their ``F.linear(x, w)`` no-bias z⁺).
- Softmax/LN/GELU/Dropout: ``Pass`` (identity, as in their relprop).
- Conv (patch-embed): unmapped — ``transformer_attribution`` reads ``R_A`` at the
  softmax and never propagates below it (pixel-space ``method="full"`` is out of scope).

## 1. Setup

In [1]:
from __future__ import annotations
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent

import timm
from crp.attribution import CondAttribution
from experiments.xai_methods import (
    chefer_transformer_attribution, softmax_layer_names, model_geometry,
)
from zennit_extensions import CheferLRPComposite

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = REPO / "data" / "chefer_reference"
FIG_DIR = REPO / "figures" / "chefer_reference" / "timm_jx_codeexact"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch: {torch.__version__}  device: {DEVICE}")
print(f"reference NPZs: {sorted(DATA_DIR.glob('*.npz'))}")

torch: 2.11.0+cu130  device: cuda
reference NPZs: [PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/catdog__cls243.npz'), PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/catdog__cls282.npz'), PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/dogbird__cls161.npz'), PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/dogbird__cls87.npz'), PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/el2__cls101.npz'), PosixPath('/home/claude/workspaces/zennit-crp/data/chefer_reference/el2__cls340.npz')]


## 2. Load timm ViT-B/16 (same checkpoint as Chefer reference)

In [2]:
backbone = timm.create_model(
    "vit_base_patch16_224.orig_in21k_ft_in1k", pretrained=True
).to(DEVICE).eval()

class ViTWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, x):
        return self.backbone(x)

model = ViTWrapper(backbone)
attribution = CondAttribution(model)
composite = CheferLRPComposite()
softmax_layers = softmax_layer_names(model)
print(f"Softmax layers: {len(softmax_layers)} (e.g. {softmax_layers[0]})")

Softmax layers: 12 (e.g. backbone.blocks.0.attn.softmax)


## 3. G0 — Forward gate

Verify the timm model produces the same logits as the Chefer reference
(same checkpoint, same architecture). The input tensor is shipped in the NPZ
so there is no preprocessing drift.

In [3]:
results = []
for npz_path in sorted(DATA_DIR.glob("*.npz")):
    ref = np.load(npz_path)
    x = torch.from_numpy(ref["x"]).to(DEVICE)
    target = int(ref["target"])
    with torch.no_grad():
        logits = model(x.unsqueeze(0))
    logits_ref = torch.from_numpy(ref["logits"]).to(DEVICE)
    logit_diff = (logits - logits_ref).abs().max().item()
    top5_ours = logits.topk(5, dim=1).indices[0].cpu().numpy().tolist()
    top5_ref = ref["top5"].tolist()
    match = top5_ours == top5_ref
    tag = "PASS" if (logit_diff < 1e-3 and match) else "FAIL"
    print(f"[{tag}] {npz_path.name:25s}  max|d-logits|={logit_diff:.2e}  top5_match={match}")
    results.append({"name": npz_path.stem, "x": x, "target": target, "ref": ref,
                    "logit_diff": logit_diff, "top5_match": match})

[PASS] catdog__cls243.npz         max|d-logits|=5.25e-06  top5_match=True
[PASS] catdog__cls282.npz         max|d-logits|=5.25e-06  top5_match=True
[PASS] dogbird__cls161.npz        max|d-logits|=3.16e-06  top5_match=True
[PASS] dogbird__cls87.npz         max|d-logits|=3.16e-06  top5_match=True
[PASS] el2__cls101.npz            max|d-logits|=3.10e-06  top5_match=True
[PASS] el2__cls340.npz            max|d-logits|=3.10e-06  top5_match=True


## 4. Reconstruction + per-block comparison

In [4]:
all_stats = []
for r in results:
    name = r["name"]
    x = r["x"]
    target = r["target"]
    ref = r["ref"]

    xn = x.unsqueeze(0)
    n_prefix, grid, patch = model_geometry(model, xn)

    heatmap, blocks = chefer_transformer_attribution(
        model, attribution, composite, xn, target,
        n_prefix=n_prefix, grid=grid,
        softmax_layers=softmax_layers, return_blocks=True,
    )

    ref_R_A = torch.from_numpy(ref["R_A"]).to(DEVICE)
    ref_G_A = torch.from_numpy(ref["G_A"]).to(DEVICE)
    block_r_errs = []
    block_g_errs = []
    for b in range(len(blocks)):
        r_err = (blocks[b][1] - ref_R_A[b].unsqueeze(0)).abs().max().item() / \
                (ref_R_A[b].abs().max().item() + 1e-12)
        g_err = (blocks[b][0] - ref_G_A[b].unsqueeze(0)).abs().max().item() / \
                (ref_G_A[b].abs().max().item() + 1e-12)
        block_r_errs.append(r_err)
        block_g_errs.append(g_err)

    our_map = heatmap[0].cpu().numpy()
    ref_map = ref["gen_map196"][0].reshape(14, 14)
    pearson = float(np.corrcoef(our_map.ravel(), ref_map.ravel())[0, 1])

    our_mask224 = torch.nn.functional.interpolate(
        torch.from_numpy(our_map).float().reshape(1, 1, 14, 14),
        scale_factor=16, mode="bilinear"
    ).reshape(224, 224).numpy()
    our_mask224 = (our_mask224 - our_mask224.min()) / (our_mask224.max() - our_mask224.min() + 1e-12)
    ref_mask224 = ref["mask224"]
    mask224_diff = np.abs(our_mask224 - ref_mask224)

    worst_r = max(block_r_errs)
    worst_g = max(block_g_errs)
    tag = "PASS" if (worst_r < 0.05 and pearson > 0.9999) else "CHECK"
    print(f"[{tag}] {name:25s}  R_A worst={worst_r:.2e}  G_A worst={worst_g:.2e}  "
          f"pearson={pearson:.6f}  mask224 max|d|={mask224_diff.max():.2e}")

    all_stats.append({
        "name": name, "target": target, "heatmap": our_map, "ref_map": ref_map,
        "block_r_errs": block_r_errs, "block_g_errs": block_g_errs,
        "pearson": pearson, "mask224_diff": mask224_diff,
        "our_mask224": our_mask224, "ref_mask224": ref_mask224,
        "ref_vis": ref["vis_bgr"], "x": x,
    })

[PASS] catdog__cls243             R_A worst=1.48e-02  G_A worst=9.22e-07  pearson=1.000000  mask224 max|d|=6.14e-06
[PASS] catdog__cls282             R_A worst=1.48e-02  G_A worst=1.60e-06  pearson=1.000000  mask224 max|d|=1.15e-05
[PASS] dogbird__cls161            R_A worst=1.62e-02  G_A worst=1.29e-06  pearson=0.999998  mask224 max|d|=3.14e-03
[PASS] dogbird__cls87             R_A worst=1.22e-02  G_A worst=9.40e-07  pearson=1.000000  mask224 max|d|=1.34e-03
[PASS] el2__cls101                R_A worst=8.28e-04  G_A worst=1.23e-06  pearson=1.000000  mask224 max|d|=1.37e-06
[PASS] el2__cls340                R_A worst=9.10e-04  G_A worst=7.69e-07  pearson=1.000000  mask224 max|d|=2.68e-07


## 5. Per-block error profile

The R_A relative error is largest at block 0 (deepest in the backward chain)
and decreases toward block 11 (nearest the seed). This is the expected
accumulation pattern from the stabilizer sign-difference: zennit's
``stabilize(x, eps)`` adds ``sign(x)*eps`` (away from zero), while Chefer's
``safe_divide`` adds ``+eps`` (toward zero for negatives). The difference is
~1e-9 per operation but compounds across 12 blocks. The final map (after
clamp + mean + rollout + minmax) washes this out to ~1e-8.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for s in all_stats:
    ax.plot(range(12), s["block_r_errs"], "o-", label=s["name"], ms=3)
ax.set_xlabel("block index")
ax.set_ylabel("R_A relative inf-error")
ax.set_title("Per-block R_A error (decreases toward seed at block 11)")
ax.legend(fontsize=6, ncol=3)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(FIG_DIR / "per_block_r_err.pdf")
plt.savefig(FIG_DIR / "per_block_r_err.png", dpi=150)
plt.show()

## 6. Visual comparison — reference vs reconstruction

In [ ]:
n = len(all_stats)
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = axes.reshape(1, -1)

# Both overlays rendered from the (correct) minmax masks with matplotlib jet.
# NOTE: the reference's own show_cam_on_image bakes a cv2 JET heatmap (BGR) onto
# an RGB image, crossing R/B — its stored ``vis_bgr`` reads temperature-inverted
# (low relevance -> red). We ignore that pre-baked overlay and colour the
# reference from its stored ``ref_mask224`` the same way as ours, so column 2 and
# column 3 use one identical, correct colormap path.
for i, s in enumerate(all_stats):
    img = s["x"].permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())
    ref_vis = 0.5 * img + 0.5 * plt.cm.jet(s["ref_mask224"])[..., :3]
    our_vis = 0.5 * img + 0.5 * plt.cm.jet(s["our_mask224"])[..., :3]

    axes[i, 0].imshow(img); axes[i, 0].axis("off")
    axes[i, 0].set_title(f"{s['name']}\ninput", fontsize=9)
    axes[i, 1].imshow(ref_vis); axes[i, 1].axis("off")
    axes[i, 1].set_title(f"reference (Chefer code)\ncls={s['target']}", fontsize=9)
    axes[i, 2].imshow(our_vis); axes[i, 2].axis("off")
    axes[i, 2].set_title(f"ours (zennit rules)\npearson={s['pearson']:.6f}", fontsize=9)
    im = axes[i, 3].imshow(s["mask224_diff"], cmap="hot")
    axes[i, 3].axis("off")
    axes[i, 3].set_title(f"|d-mask224| (max={s['mask224_diff'].max():.1e})", fontsize=9)
    plt.colorbar(im, ax=axes[i, 3], fraction=0.046)

fig.suptitle("Chefer CVPR'21 transformer_attribution - reference vs zennit code-exact",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison.pdf", bbox_inches="tight")
plt.savefig(FIG_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary

In [7]:
print(f"{'name':25s} {'target':>7s}  {'R_A worst':>10s}  {'G_A worst':>10s}  "
      f"{'pearson':>8s}  {'mask max|d|':>12s}  {'gate':>5s}")
print("-" * 90)
for s in all_stats:
    worst_r = max(s["block_r_errs"])
    worst_g = max(s["block_g_errs"])
    gate = "PASS" if (worst_r < 0.05 and s["pearson"] > 0.9999) else "CHECK"
    print(f"{s['name']:25s} {s['target']:7d}  {worst_r:10.2e}  {worst_g:10.2e}  "
          f"{s['pearson']:8.6f}  {s['mask224_diff'].max():12.2e}  {gate:>5s}")

name                       target   R_A worst   G_A worst   pearson   mask max|d|   gate
------------------------------------------------------------------------------------------
catdog__cls243                243    1.48e-02    9.22e-07  1.000000      6.14e-06   PASS
catdog__cls282                282    1.48e-02    1.60e-06  1.000000      1.15e-05   PASS
dogbird__cls161               161    1.62e-02    1.29e-06  0.999998      3.14e-03   PASS
dogbird__cls87                 87    1.22e-02    9.40e-07  1.000000      1.34e-03   PASS
el2__cls101                   101    8.28e-04    1.23e-06  1.000000      1.37e-06   PASS
el2__cls340                   340    9.10e-04    7.69e-07  1.000000      2.68e-07   PASS
